# LeetCode 3474 — Lexicographically Smallest Generated String bằng Python

- Link: https://leetcode.com/problems/lexicographically-smallest-generated-string/
- Độ khó: **Hard**
- Chủ đề: **Tham lam** (đặt cứng các vị trí `T`, điền `'a'`, sửa vị trí tự do **cuối cùng** thành `'b'` khi cửa sổ `F` bị trùng), **so khớp chuỗi**, kiểm chứng bằng **DP trên automaton KMP**

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def generateString(self, str1: str, str2: str) -> str:
```

## 1. Mục tiêu học

1. Biến điều kiện `T` thành **ràng buộc cứng** trên từng ký tự và phát hiện mâu thuẫn.
2. Hiểu vì sao ký tự tự do chỉ cần chọn trong `{'a', 'b'}` và vì sao bắt đầu bằng `'a'` hết.
3. Chứng minh chiến lược "cửa sổ `F` bị trùng ⇒ đổi vị trí tự do **phải nhất** trong cửa sổ thành `'b'`" cho xâu nhỏ nhất theo thứ tự từ điển.
4. Viết lời giải kiểm chứng tổng quát: **DP khả thi từ phải sang** trên automaton KMP rồi dựng xâu nhỏ nhất từ trái sang.
5. Đối chiếu với vét cạn trên bảng chữ cái nhỏ.

## 2. Đề bài & ví dụ

Cho `str1` (độ dài `n`, chỉ gồm `'T'`/`'F'`) và `str2` (độ dài `m`). Xâu `word` độ dài `n + m − 1` là **sinh bởi** `str1, str2` nếu với mọi `i`:

- `str1[i] = 'T'` ⇒ `word[i..i+m−1] == str2`;
- `str1[i] = 'F'` ⇒ `word[i..i+m−1] != str2`.

Trả về xâu sinh được **nhỏ nhất theo thứ tự từ điển**, hoặc `""` nếu không có.

**Ví dụ 1:** `"TFTF", "ab"` → `"ababa"`. **Ví dụ 2:** `"TFTF", "abc"` → `""`. **Ví dụ 3:** `"F", "d"` → `"a"`.

**Ràng buộc:** `1 ≤ n ≤ 10^4`, `1 ≤ m ≤ 500`.

In [1]:
examples = [("TFTF", "ab", "ababa"), ("TFTF", "abc", ""), ("F", "d", "a")]

def is_generated(word, s1, s2):
    m = len(s2)
    if len(word) != len(s1) + m - 1:
        return False
    return all((word[i:i + m] == s2) == (c == 'T') for i, c in enumerate(s1))

for s1, s2, exp in examples:
    print(s1, s2, '-> kỳ vọng', repr(exp), '| kiểm tra:', is_generated(exp, s1, s2) if exp else 'không có xâu')

TFTF ab -> kỳ vọng 'ababa' | kiểm tra: True
TFTF abc -> kỳ vọng '' | kiểm tra: không có xâu
F d -> kỳ vọng 'a' | kiểm tra: True


## 3. Kiến thức nền

### 3.1. Thứ tự từ điển và tham lam từ trái sang

Xâu `x < y` (cùng độ dài) ⇔ tại vị trí khác nhau đầu tiên, `x` có ký tự nhỏ hơn. ⇒ Muốn nhỏ nhất: giữ các vị trí **đầu** nhỏ nhất có thể, chấp nhận "hy sinh" ở vị trí **càng về sau càng tốt**.

### 3.2. Đặt các cửa sổ `T` — phát hiện mâu thuẫn

Mỗi `T` ở `i` ép `word[i + j] = str2[j]`. Hai cửa sổ `T` chồng nhau phải đồng ý tại phần giao.

In [2]:
def place_T(s1, s2):
    n, m = len(s1), len(s2)
    w = [None] * (n + m - 1)
    for i, c in enumerate(s1):
        if c == 'T':
            for j in range(m):
                if w[i + j] is None:
                    w[i + j] = s2[j]
                elif w[i + j] != s2[j]:
                    return None, (i, i + j)
    return w, None

print(place_T("TFTF", "ab"))
print(place_T("TFTF", "abc"))

(['a', 'b', 'a', 'b', None], None)
(None, (2, 2))


### 3.3. Hàm tiền tố (KMP) và automaton

`pi[i]` = độ dài **biên** dài nhất (vừa là tiền tố vừa là hậu tố thật) của `str2[0..i]`. Automaton: trạng thái = độ dài tiền tố của `str2` đang khớp ở cuối; đọc thêm ký tự ⇒ trạng thái mới. Trạng thái `m` ⇔ vừa kết thúc một lần xuất hiện của `str2`.

In [3]:
def prefix_function(p):
    pi = [0] * len(p)
    for i in range(1, len(p)):
        k = pi[i - 1]
        while k and p[i] != p[k]:
            k = pi[k - 1]
        if p[i] == p[k]:
            k += 1
        pi[i] = k
    return pi

def make_step(p):
    m, pi = len(p), prefix_function(p)
    def step(st, ch):
        if st == m:
            st = pi[m - 1]
        while st and p[st] != ch:
            st = pi[st - 1]
        return st + 1 if p[st] == ch else 0
    return step

print(prefix_function("abab"), prefix_function("aabaaab"))
step = make_step("ab")
st, states = 0, []
for ch in "ababa":
    st = step(st, ch); states.append(st)
print('trạng thái khi đọc "ababa" với mẫu "ab":', states)

[0, 0, 1, 2] [0, 1, 0, 1, 2, 2, 3]
trạng thái khi đọc "ababa" với mẫu "ab": [1, 2, 1, 2, 1]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

1. Đặt cứng các cửa sổ `T`. Mâu thuẫn ⇒ `""`.
2. Mọi vị trí còn lại (**tự do**) điền `'a'` — nhỏ nhất có thể.
3. Duyệt các cửa sổ `F` từ trái sang. Nếu cửa sổ đang **trùng** `str2`, phải đổi ít nhất một ký tự **tự do** trong cửa sổ (ký tự cố định không được đổi). Đổi cái **phải nhất** thành `'b'` (nhỏ nhất khác `'a'`), vì đổi càng về sau càng ít làm xâu "lớn lên". Không có vị trí tự do ⇒ `""`.

Tính tay ví dụ 1 (`"TFTF"`, `"ab"`): đặt `T` tại 0 và 2 ⇒ `a b a b _`. Vị trí 4 tự do ⇒ `'a'` ⇒ `ababa`. Cửa sổ `F` tại 1: `"ba"` ≠ `"ab"` ✓. Tại 3: `"ba"` ✓ ⇒ `"ababa"`.

Ví dụ 3 (`"F"`, `"d"`): vị trí 0 tự do ⇒ `'a'`; cửa sổ `"a"` ≠ `"d"` ✓ ⇒ `"a"`.

Vì sao chỉ cần `'b'`: mẫu `str2` đòi một ký tự cụ thể tại mỗi vị trí; chỉ cần ký tự tại đó **khác** là đủ. Đang là `'a'` mà trùng ⇒ `str2` cần `'a'` ở đó ⇒ `'b'` chắc chắn khác.

In [4]:
w, _ = place_T("TFTF", "ab")
print('sau khi đặt T:', w)
print('điền a:', ''.join(c if c else 'a' for c in w))

sau khi đặt T: ['a', 'b', 'a', 'b', None]
điền a: ababa


## 5. Cách 1 — Brute Force: thử mọi xâu theo thứ tự từ điển

Duyệt mọi xâu trên một bảng chữ cái nhỏ (đủ chứa các ký tự của `str2` và thêm `'a'`, `'b'`) theo thứ tự từ điển; xâu hợp lệ đầu tiên là đáp án.

In [5]:
import itertools

def generate_brute(s1, s2):
    alpha = sorted(set(s2) | {'a', 'b'})
    L = len(s1) + len(s2) - 1
    for t in itertools.product(alpha, repeat=L):
        w = ''.join(t)
        if is_generated(w, s1, s2):
            return w
    return ''

for s1, s2, exp in examples:
    print(repr(generate_brute(s1, s2)), '| kỳ vọng', repr(exp))

'ababa' | kỳ vọng 'ababa'
'' | kỳ vọng ''
'a' | kỳ vọng 'a'


**Độ phức tạp:** `O(|Σ|^{n+m−1} · n · m)` — chỉ dùng cho `n + m ≤ 8`.

(Bảng chữ cái `set(s2) ∪ {a, b}` là đủ: một ký tự không có trong `str2` và lớn hơn `'b'` có thể thay bằng `'a'` hoặc `'b'` mà không phá điều kiện nào — xem mục 10.)

## 6. Ý tưởng thuật toán / bước đột phá

**Tham lam "phải nhất":** sau bước đặt `T` và điền `'a'`, duyệt `i` tăng dần với `str1[i] = 'F'`:

- Nếu `word[i..i+m−1] == str2`: tìm `j` lớn nhất trong `[i, i+m−1]` là vị trí tự do **chưa bị đổi**; đặt `word[j] = 'b'`, đánh dấu `j` đã cố định. Không có ⇒ `""`.

**Hai điều cần thấy:**

1. **Tính hợp lệ:** đổi `j` không được làm một cửa sổ `F` phía trước (`i' < i`) trở thành trùng. Mục 10 chứng minh một phần then chốt, phần còn lại được kiểm chứng bằng máy (mục 12 và bài tập 5).
2. Đổi vị trí phải nhất là tối ưu: mọi xâu hợp lệ phải khác `'a'` ở **ít nhất một** vị trí tự do của cửa sổ này; chọn phải nhất cho tiền tố dài nhất toàn `'a'`.

**Cách kiểm chứng tổng quát (theo gợi ý đề):** DP `ok[p][st]` = từ vị trí `p` với trạng thái automaton `st` còn hoàn thành được không (ký tự tự do ∈ `{a, b}`, cấm đạt trạng thái `m` tại điểm kết thúc của cửa sổ `F`). Sau đó dựng xâu từ trái, mỗi bước chọn ký tự nhỏ nhất vẫn khả thi.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tham lam "phải nhất" (lời giải chính)

In [6]:
def generate_greedy(s1, s2):
    n, m = len(s1), len(s2)
    w, conflict = place_T(s1, s2)
    if w is None:
        return ''
    free = [c is None for c in w]
    w = [c if c is not None else 'a' for c in w]
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            for j in range(i + m - 1, i - 1, -1):
                if free[j]:
                    w[j] = 'b'
                    free[j] = False
                    break
            else:
                return ''
    return ''.join(w)

for s1, s2, exp in examples:
    print(repr(generate_greedy(s1, s2)), '| kỳ vọng', repr(exp))

'ababa' | kỳ vọng 'ababa'
'' | kỳ vọng ''
'a' | kỳ vọng 'a'


### 7.2. Cách 3 — DP khả thi trên automaton KMP + dựng xâu nhỏ nhất

Trạng thái `(p, st)`: `(n + m − 1) · (m + 1)` ≈ `5·10^6` ở cỡ lớn nhất ⇒ trong Python khá chậm (vài giây đến chục giây), nhưng là **lời giải kiểm chứng độc lập** rất tốt vì không cần lập luận tham lam.

In [7]:
def generate_dp(s1, s2):
    n, m = len(s1), len(s2)
    L = n + m - 1
    w, _ = place_T(s1, s2)
    if w is None:
        return ''
    step = make_step(s2)
    forbid = [False] * L                  # tại vị trí kết thúc của cửa sổ F, cấm trạng thái m
    for i, c in enumerate(s1):
        if c == 'F':
            forbid[i + m - 1] = True
    cand = [(w[p],) if w[p] is not None else ('a', 'b') for p in range(L)]
    ok = [None] * (L + 1)
    ok[L] = [True] * (m + 1)
    for p in range(L - 1, -1, -1):
        nxt, row = ok[p + 1], [False] * (m + 1)
        for st in range(m + 1):
            for ch in cand[p]:
                ns = step(st, ch)
                if ns == m and forbid[p]:
                    continue
                if nxt[ns]:
                    row[st] = True
                    break
        ok[p] = row
    if not ok[0][0]:
        return ''
    st, out = 0, []
    for p in range(L):
        for ch in cand[p]:
            ns = step(st, ch)
            if ns == m and forbid[p]:
                continue
            if ok[p + 1][ns]:
                out.append(ch); st = ns
                break
    return ''.join(out)

for s1, s2, exp in examples:
    print(repr(generate_dp(s1, s2)), '| kỳ vọng', repr(exp))

'ababa' | kỳ vọng 'ababa'
'' | kỳ vọng ''
'a' | kỳ vọng 'a'


Lưu ý: điều kiện `T` đã được bảo đảm bởi ký tự cố định, nên DP chỉ cần cấm "khớp" ở cửa sổ `F`. Vị trí `T` có thể dẫn tới trạng thái `m` — hợp lệ.

### 7.3. Cách 4 — Tham lam + kiểm tra mâu thuẫn `T` bằng hàm Z

Hai cửa sổ `T` ở `i < i'` với `d = i' − i < m` tương thích ⇔ `str2` có **chu kỳ** `d` ⇔ `z[d] ≥ m − d`. Chỉ cần kiểm tra **cặp `T` liên tiếp** (tính tương thích lan truyền qua chuỗi các cửa sổ chồng nhau). Sau đó đặt ký tự bằng con trỏ "đã phủ tới đâu" để mỗi ký tự chỉ ghi một lần — bước đặt `T` thành `O(n + m)`.

In [8]:
def z_function(s):
    n = len(s)
    z = [0] * n
    if n:
        z[0] = n
    l = r = 0
    for i in range(1, n):
        if i < r:
            z[i] = min(r - i, z[i - l])
        while i + z[i] < n and s[z[i]] == s[i + z[i]]:
            z[i] += 1
        if i + z[i] > r:
            l, r = i, i + z[i]
    return z

def generate_greedy_z(s1, s2):
    n, m = len(s1), len(s2)
    L = n + m - 1
    z = z_function(s2)
    Ts = [i for i, c in enumerate(s1) if c == 'T']
    for a, b in zip(Ts, Ts[1:]):
        d = b - a
        if d < m and z[d] < m - d:
            return ''
    w = ['a'] * L
    free = [True] * L
    covered = 0                               # đã ghi tới vị trí covered - 1
    for i in Ts:
        for p in range(max(i, covered), i + m):
            w[p] = s2[p - i]
            free[p] = False
        covered = max(covered, i + m)
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            for j in range(i + m - 1, i - 1, -1):
                if free[j]:
                    w[j] = 'b'; free[j] = False
                    break
            else:
                return ''
    return ''.join(w)

print(z_function("abab"), z_function("aaa"))
for s1, s2, exp in examples:
    print(repr(generate_greedy_z(s1, s2)), '| kỳ vọng', repr(exp))

[4, 0, 2, 0] [3, 2, 1]
'ababa' | kỳ vọng 'ababa'
'' | kỳ vọng ''
'a' | kỳ vọng 'a'


## 8. TRACE từng bước

Trace cách 2 với `str1 = "FFFTF"`, `str2 = "aa"`: cửa sổ `T` ở 3 cố định vị trí 3, 4; các cửa sổ `F` trùng mẫu buộc phải đổi.

In [9]:
def trace_greedy(s1, s2):
    n, m = len(s1), len(s2)
    w, conflict = place_T(s1, s2)
    if w is None:
        print('mâu thuẫn T tại', conflict); return ''
    print('sau khi đặt T :', ''.join(c if c else '_' for c in w))
    free = [c is None for c in w]
    w = [c if c is not None else 'a' for c in w]
    print('điền a        :', ''.join(w))
    for i, c in enumerate(s1):
        if c != 'F':
            continue
        window = ''.join(w[i:i + m])
        if window != s2:
            print(f"F tại {i}: cửa sổ {window!r} != {s2!r} -> bỏ qua")
            continue
        for j in range(i + m - 1, i - 1, -1):
            if free[j]:
                w[j] = 'b'; free[j] = False
                print(f"F tại {i}: cửa sổ {window!r} trùng -> đổi vị trí {j} thành 'b' -> {''.join(w)}")
                break
        else:
            print(f"F tại {i}: trùng nhưng không còn vị trí tự do -> ''"); return ''
    res = ''.join(w)
    print('kết quả:', res, '| hợp lệ?', is_generated(res, s1, s2), '| vét cạn:', generate_brute(s1, s2))
    return res

trace_greedy("FFFTF", "aa")

sau khi đặt T : ___aa_
điền a        : aaaaaa
F tại 0: cửa sổ 'aa' trùng -> đổi vị trí 1 thành 'b' -> abaaaa
F tại 1: cửa sổ 'ba' != 'aa' -> bỏ qua
F tại 2: cửa sổ 'aa' trùng -> đổi vị trí 2 thành 'b' -> abbaaa
F tại 4: cửa sổ 'aa' trùng -> đổi vị trí 5 thành 'b' -> abbaab
kết quả: abbaab | hợp lệ? True | vét cạn: abbaab


'abbaab'

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def generateString(self, str1: str, str2: str) -> str:
        n, m = len(str1), len(str2)
        L = n + m - 1
        w = [''] * L
        for i, c in enumerate(str1):
            if c == 'T':
                for j in range(m):
                    if w[i + j] == '':
                        w[i + j] = str2[j]
                    elif w[i + j] != str2[j]:
                        return ''
        free = [c == '' for c in w]
        for p in range(L):
            if free[p]:
                w[p] = 'a'
        for i, c in enumerate(str1):
            if c == 'F' and ''.join(w[i:i + m]) == str2:
                j = i + m - 1
                while j >= i and not free[j]:
                    j -= 1
                if j < i:
                    return ''
                w[j] = 'b'
                free[j] = False
        return ''.join(w)

for s1, s2, exp in examples:
    got = Solution().generateString(s1, s2)
    print(repr(got), 'OK' if got == exp else 'SAI')

'ababa' OK
'' OK
'a' OK


## 10. Vì sao thuật toán đúng

Gọi `C` là tập vị trí cố định bởi `T`, `U` là tập vị trí tự do.

1. **Bước `T`:** mọi xâu hợp lệ phải trùng các ký tự đặt ở `C`; mâu thuẫn ⇒ không có xâu nào.
2. **Bất biến của tham lam:** sau khi xử lý xong cửa sổ `F` tại `i`, mọi cửa sổ `F` tại `i' ≤ i` khác `str2`, và tập `B` các vị trí đã đổi thành `'b'` là **nhỏ nhất theo nghĩa từ điển** (mọi xâu hợp lệ `x` đều có, với mỗi `j ∈ B`, … ). Lập luận chuẩn: xét cửa sổ `i` đang trùng. Mọi xâu hợp lệ `x` khớp tiền tố `word[0..j−1]` (toàn `'a'` ở các vị trí tự do chưa đổi) phải có một vị trí tự do trong `[i, i+m−1]` khác ký tự hiện tại; các vị trí tự do trong `(j, i+m−1]` **không tồn tại** (vì `j` phải nhất) ⇒ `x` phải khác tại một vị trí ≤ `j`; muốn giữ tiền tố `[0, j)` thì phải khác tại đúng `j`, nhỏ nhất là `'b'`. Do đó xâu tham lam là nhỏ nhất **nếu nó hợp lệ**.
3. **Đổi `j` không phá cửa sổ `F` trước đó (`i' < i`, cửa sổ `i'` chứa `j`).**
   - *Bước 1 — lúc xử lý `i'`, cửa sổ `i'` không trùng.* Nếu nó trùng thì ta đã đổi vị trí tự do phải nhất `j'` của nó thành `'b'`. Lúc đó `j` còn tự do và nằm trong cửa sổ `i'` nên `j' ≥ j`, mà `j` vẫn chưa bị đổi nên `j' > j`. Khi đó cửa sổ `i'` trùng với ký tự `'a'` tại `j'` ⇒ `str2[j' − i'] = 'a'`, nhưng bây giờ `word[j'] = 'b'` ⇒ cửa sổ `i'` không thể trùng nữa (mỗi vị trí chỉ đổi một lần). Mâu thuẫn.
   - *Bước 2 — vậy cửa sổ `i'` có các vị trí lệch* (đều phải là `'a'` tự do mà `str2` cần `'b'`, vì sau này trùng được). Để cửa sổ `i'` trùng sau khi đổi `j`, mọi vị trí lệch đó đều đã bị đổi, và `j` là vị trí cuối cùng. Các vị trí của cửa sổ `i'` nằm bên phải `j` đều thuộc cửa sổ `i` nên **không tự do** lúc xử lý `i` ⇒ chúng được cố định bởi các cửa sổ `T` bắt đầu **sau** `j` (cửa sổ `T` bắt đầu ≤ `j` sẽ phủ cả `j`, mà `j` tự do). Kết hợp với việc cả cửa sổ `i` và cửa sổ `i'` (sau khi đổi) đều bằng `str2`, ta được các ràng buộc chu kỳ trên `str2` mâu thuẫn với các ký tự cố định đó — ví dụ nhỏ nhất kiểu `word = "aabb"`, `str2 = "abb"` sẽ đòi vị trí 2, 3 cố định mà không có cửa sổ `T` nào phủ được chúng mà không phủ vị trí 1.
   - Phần tổng quát của bước 2 là bài tập 5; notebook xác nhận bằng **hai lời giải độc lập** (DP automaton không dựa vào tham lam, và vét cạn) cùng hàm `check_hypothesis` (0 vi phạm trên 20.000 bộ).
4. **Bảng chữ cái `{a, b}` là đủ:** ở vị trí tự do, thay ký tự `c ∉ {a, b}` bằng `'a'` hoặc `'b'` (chọn cái làm cửa sổ chứa nó vẫn khác `str2`) không làm hỏng điều kiện `F` nào mà chỉ làm xâu nhỏ hơn — DP (cách 3) dùng giả định này và vét cạn trên `set(str2) ∪ {a, b}` xác nhận.

Ghi chú trung thực: mục 2 (tối ưu) là chặt chẽ khi xâu tham lam hợp lệ; mục 3 (hợp lệ) có bước 1 chặt chẽ, bước 2 được kiểm chứng bằng máy.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(|Σ|^{n+m−1} n m)` | `O(n + m)` |
| 2. Tham lam | `O(n · m)` (đặt `T` + so sánh cửa sổ) | `O(n + m)` |
| 3. DP automaton | `O((n + m) · m · 2 · α)` (α: số bước lùi KMP) | `O((n + m) · m)` |
| 4. Tham lam + Z | đặt `T` `O(n + m)`, so sánh `F` vẫn `O(n · m)` tệ nhất | `O(n + m)` |

Với `n = 10^4, m = 500`: `n · m = 5·10^6` phép so sánh ký tự (so sánh lát cắt chuỗi trong Python chạy ở C nên rất nhanh).

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'greedy': generate_greedy, 'dp': generate_dp, 'greedy_z': generate_greedy_z,
         'submit': lambda a, b: Solution().generateString(a, b)}
edge = [
    ("T", "z", "z"),
    ("F", "a", "b"),                 # 'a' trùng mẫu -> 'b'
    ("FF", "a", "bb"),
    ("TT", "ab", ""),                # hai cửa sổ T chồng nhau mâu thuẫn
    ("TT", "aa", "aaa"),             # chồng nhau nhưng tương thích (chu kỳ 1)
    ("FT", "a", ""),                 # vị trí 0 phải khác 'a', vị trí 1 = 'a' -> 'ba'? kiểm tra dưới
]
edge[-1] = ("FT", "a", "ba")
for s1, s2, exp in examples + edge:
    assert generate_brute(s1, s2) == exp, (s1, s2, generate_brute(s1, s2))
    for name, f in impls.items():
        assert f(s1, s2) == exp, (name, s1, s2)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (và kiểm tra tính hợp lệ)

In [12]:
import random
rng = random.Random(3474)
cnt = 0
for _ in range(2500):
    n = rng.randint(1, 5)
    m = rng.randint(1, 3)
    if n + m - 1 > 6:
        continue
    s1 = ''.join(rng.choice('TF') for _ in range(n))
    s2 = ''.join(rng.choice(rng.choice(['ab', 'a', 'abc', 'bc'])) for _ in range(m))
    exp = generate_brute(s1, s2)
    for name, f in impls.items():
        got = f(s1, s2)
        assert got == exp, (name, s1, s2, got, exp)
        assert got == '' or is_generated(got, s1, s2)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

# so sánh tham lam với DP ở cỡ vừa (vét cạn không chạy nổi)
for _ in range(300):
    n = rng.randint(1, 40); m = rng.randint(1, 6)
    s1 = ''.join(rng.choice('TFFF') for _ in range(n))
    s2 = ''.join(rng.choice('ab') for _ in range(m))
    g = generate_greedy(s1, s2)
    assert g == generate_dp(s1, s2) and (g == '' or is_generated(g, s1, s2))
print('Tham lam khớp DP trên 300 bộ cỡ vừa.')

Khớp vét cạn trên 2333 bộ ngẫu nhiên.
Tham lam khớp DP trên 300 bộ cỡ vừa.


### 12.3. Ca lớn `n = 10^4`, `m = 500`

In [13]:
import time
rng = random.Random(1)
big = {
    'toàn F, mẫu toàn a': ('F' * 10 ** 4, 'a' * 500),
    'T thưa, mẫu chu kỳ': (''.join('T' if i % 700 == 0 else 'F' for i in range(10 ** 4)), 'ab' * 250),
    'T dày, mẫu toàn a': (''.join(rng.choice('TF') for _ in range(10 ** 4)), 'a' * 500),
    'ngẫu nhiên, mẫu a..ab': (''.join(rng.choice('FFFFT') for _ in range(10 ** 4)), 'a' * 499 + 'b'),
}
for name, (s1, s2) in big.items():
    t = time.time(); r = Solution().generateString(s1, s2); t1 = time.time() - t
    t = time.time(); r4 = generate_greedy_z(s1, s2); t4 = time.time() - t
    assert r == r4 and (r == '' or is_generated(r, s1, s2))
    print(f"{name:24s}: độ dài kết quả {len(r):5d}, số 'b' = {r.count('b'):5d} | tham lam {t1:.2f}s | +Z {t4:.2f}s")

s1, s2 = big['toàn F, mẫu toàn a']
s1, s2 = s1[:3000], s2[:200]
t = time.time(); r_dp = generate_dp(s1, s2); t_dp = time.time() - t
assert r_dp == Solution().generateString(s1, s2)
print(f"DP automaton ở cỡ n=3000, m=200: {t_dp:.2f}s (cỡ tối đa sẽ chậm hơn nhiều lần — chỉ dùng để kiểm chứng)")

toàn F, mẫu toàn a      : độ dài kết quả 10499, số 'b' =    20 | tham lam 0.02s | +Z 0.02s


T thưa, mẫu chu kỳ      : độ dài kết quả 10499, số 'b' =  3750 | tham lam 0.02s | +Z 0.02s


T dày, mẫu toàn a       : độ dài kết quả     0, số 'b' =     0 | tham lam 0.11s | +Z 0.00s
ngẫu nhiên, mẫu a..ab   : độ dài kết quả     0, số 'b' =     0 | tham lam 0.00s | +Z 0.00s


DP automaton ở cỡ n=3000, m=200: 0.11s (cỡ tối đa sẽ chậm hơn nhiều lần — chỉ dùng để kiểm chứng)


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định, so với vét cạn):

In [14]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(1, 5); m = rng.randint(1, 3)
        if n + m - 1 > 6:
            continue
        s1 = ''.join(rng.choice('TF') for _ in range(n))
        s2 = ''.join(rng.choice('ab') for _ in range(m))
        if wrong(s1, s2) != generate_brute(s1, s2):
            return s1, s2
    return None

### Lỗi 1 — Đổi vị trí tự do **trái nhất** trong cửa sổ

In [15]:
def wrong_leftmost(s1, s2):
    n, m = len(s1), len(s2)
    w, _ = place_T(s1, s2)
    if w is None:
        return ''
    free = [c is None for c in w]
    w = [c if c is not None else 'a' for c in w]
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            for j in range(i, i + m):            # SAI: trái nhất
                if free[j]:
                    w[j] = 'b'; free[j] = False
                    break
            else:
                return ''
    return ''.join(w)

ce = find_counterexample(wrong_leftmost, seed=1)
print('phản ví dụ:', ce)
print('sai :', repr(wrong_leftmost(*ce)), '| đúng:', repr(generate_brute(*ce)))

phản ví dụ: ('F', 'aa')
sai : 'ba' | đúng: 'ab'


**Vì sao sai:** đổi ở vị trí sớm làm xâu lớn lên sớm hơn; vị trí phải nhất cũng phá được cửa sổ mà giữ tiền tố nhỏ hơn.

### Lỗi 2 — Quên xử lý cửa sổ `F` (điền `'a'` là xong)

In [16]:
def wrong_ignore_F(s1, s2):
    w, _ = place_T(s1, s2)
    if w is None:
        return ''
    return ''.join(c if c is not None else 'a' for c in w)

print('sai :', repr(wrong_ignore_F("F", "a")), '-> hợp lệ?', is_generated(wrong_ignore_F("F", "a"), "F", "a"),
      '| đúng:', repr(Solution().generateString("F", "a")))

sai : 'a' -> hợp lệ? False | đúng: 'b'


**Vì sao sai:** điều kiện `F` là "**khác** `str2`", điền toàn `'a'` có thể vô tình tạo ra đúng `str2`.

### Lỗi 3 — Ghi đè cửa sổ `T` mà không kiểm tra mâu thuẫn

In [17]:
def wrong_overwrite(s1, s2):
    n, m = len(s1), len(s2)
    w = ['a'] * (n + m - 1)
    free = [True] * (n + m - 1)
    for i, c in enumerate(s1):
        if c == 'T':
            for j in range(m):
                w[i + j] = s2[j]; free[i + j] = False     # SAI: ghi đè, không báo mâu thuẫn
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            for j in range(i + m - 1, i - 1, -1):
                if free[j]:
                    w[j] = 'b'; free[j] = False
                    break
            else:
                return ''
    return ''.join(w)

r = wrong_overwrite("TFTF", "abc")
print('sai :', repr(r), '-> hợp lệ?', is_generated(r, "TFTF", "abc"), '| đúng:', repr(Solution().generateString("TFTF", "abc")))

sai : 'ababca' -> hợp lệ? False | đúng: ''


**Vì sao sai:** cửa sổ `T` ở 0 và 2 với `"abc"` đòi vị trí 2 vừa là `'c'` vừa là `'a'`. Ghi đè làm cửa sổ `T` đầu tiên không còn bằng `str2`.

### Lỗi 4 — Luôn đổi ký tự **cuối cửa sổ** mà không xét nó có tự do hay không

In [18]:
def wrong_last_char(s1, s2):
    n, m = len(s1), len(s2)
    w, _ = place_T(s1, s2)
    if w is None:
        return ''
    w = [c if c is not None else 'a' for c in w]
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            j = i + m - 1                        # SAI: có thể là ký tự do cửa sổ T cố định
            w[j] = 'b' if w[j] != 'b' else 'c'
    return ''.join(w)

ce = find_counterexample(wrong_last_char, seed=4)
print('phản ví dụ:', ce)
r = wrong_last_char(*ce)
print('sai :', repr(r), '-> hợp lệ?', is_generated(r, *ce), '| đúng:', repr(generate_brute(*ce)))

phản ví dụ: ('TTFTF', 'bb')
sai : 'bbbcba' -> hợp lệ? False | đúng: ''


**Vì sao sai:** ký tự cuối cửa sổ có thể thuộc một cửa sổ `T` phía sau; đổi nó làm cửa sổ `T` đó không còn bằng `str2`. Chỉ được đổi vị trí **tự do**.

(Ghi chú: một biến thể khác — đổi thành `'b'` nhưng quên đánh dấu vị trí đã đổi — tìm bằng máy trên 200.000 bộ ngẫu nhiên không thấy phản ví dụ, vì cửa sổ sau trùng mẫu luôn có vị trí tự do mới nằm bên phải vị trí đã đổi. Vẫn nên đánh dấu để code rõ ràng.)

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Xâu nhỏ nhất theo từ điển thỏa các ràng buộc cửa sổ" ⇒ đặt ràng buộc cứng trước, điền `'a'`, sửa **muộn nhất** có thể.
- Ràng buộc "khác mẫu" chỉ cần **một** vị trí lệch ⇒ `'b'` là đủ.
- Khi không chắc tham lam đúng ⇒ viết DP automaton KMP để đối chiếu (gợi ý của đề).
- Tương thích của hai lần xuất hiện chồng nhau ⇔ chu kỳ của mẫu ⇔ hàm Z / hàm tiền tố.

## 15. Biến thể của bài

1. Đếm số xâu sinh được (bảng chữ cái 26) ⇒ DP automaton với 26 ký tự, đếm theo modulo.
2. Xâu **lớn nhất** theo từ điển ⇒ điền `'z'`, đổi phải nhất thành `'y'`.
3. Nhiều mẫu khác nhau ⇒ automaton Aho–Corasick.
4. Bài liên quan: LC 1392 (Longest Happy Prefix — hàm tiền tố), LC 28 (Find the Index of the First Occurrence).

In [19]:
# Biến thể 2: xâu lớn nhất theo từ điển — đối xứng hoàn toàn
def generate_largest(s1, s2):
    n, m = len(s1), len(s2)
    w, _ = place_T(s1, s2)
    if w is None:
        return ''
    free = [c is None for c in w]
    w = [c if c is not None else 'z' for c in w]
    for i, c in enumerate(s1):
        if c == 'F' and ''.join(w[i:i + m]) == s2:
            for j in range(i + m - 1, i - 1, -1):
                if free[j]:
                    w[j] = 'y'; free[j] = False
                    break
            else:
                return ''
    return ''.join(w)

def largest_brute(s1, s2):
    alpha = sorted(set(s2) | {'y', 'z'}, reverse=True)
    for t in itertools.product(alpha, repeat=len(s1) + len(s2) - 1):
        w = ''.join(t)
        if is_generated(w, s1, s2):
            return w
    return ''

rng = random.Random(15)
for _ in range(600):
    n = rng.randint(1, 4); m = rng.randint(1, 3)
    s1 = ''.join(rng.choice('TF') for _ in range(n)); s2 = ''.join(rng.choice('yz') for _ in range(m))
    assert generate_largest(s1, s2) == largest_brute(s1, s2), (s1, s2)
print('Biến thể 2 khớp vét cạn:', generate_largest("TFTF", "zy"))

Biến thể 2 khớp vét cạn: zyzyz


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn theo thứ tự từ điển | mũ | Oracle |
| 2 | Tham lam "phải nhất" | `O(n m)` | **Nộp bài** |
| 3 | DP automaton KMP | `O((n+m) m)` | Kiểm chứng độc lập, chậm trong Python |
| 4 | Tham lam + Z | `O(n m)` tệ nhất | Đặt `T` tuyến tính |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `str1 = "FF"`, `str2 = "ab"`.

**Bài 2 (tính tay):** `str1 = "FFF"`, `str2 = "aa"`.

**Bài 3 (điền TODO):** viết `count_generated(s1, s2)` — đếm số xâu sinh được trên bảng chữ cái `{'a', 'b'}` (DP automaton, đếm thay vì khả thi).

**Bài 4 (tự viết lại):** viết `t_compatible(s1, s2)` chỉ dùng hàm tiền tố (không dùng Z) để kiểm tra các cửa sổ `T` có tương thích.

**Bài 5 (mini challenge):** chứng minh (hoặc tìm phản ví dụ bằng máy cho giả thuyết) "đổi vị trí phải nhất không bao giờ làm một cửa sổ `F` phía trước trở thành trùng".

In [20]:
def count_generated_todo(s1, s2):
    # TODO
    raise NotImplementedError

def t_compatible_todo(s1, s2):
    # TODO
    raise NotImplementedError

def check_hypothesis_todo(trials):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Độ dài 3, điền `"aaa"`. (2) Cửa sổ `"aa"` ≠ `"ab"`. (3) Không cần đổi.

**Bài 2.** (1) Độ dài 4, `"aaaa"`. (2) Cửa sổ 0 trùng ⇒ đổi vị trí 1. (3) Kiểm tra tiếp cửa sổ 1, 2.

**Bài 3.** (1) `cnt[p][st]` thay cho `ok[p][st]`. (2) Cộng qua các ký tự hợp lệ. (3) Đối chiếu vét cạn trên `{a, b}`.

**Bài 4.** (1) Chu kỳ `d` ⇔ biên độ dài `m − d`. (2) Tập các biên = chuỗi `pi[m−1], pi[pi[m−1]−1], …`. (3) Chỉ xét cặp `T` liên tiếp.

**Bài 5.** (1) Mô phỏng tham lam và sau **mỗi** lần đổi kiểm tra lại mọi cửa sổ `F` đã xử lý. (2) Chạy hàng nghìn bộ ngẫu nhiên. (3) Nếu không tìm được phản ví dụ, thử chứng minh bằng cách nhìn vị trí lệch của cửa sổ cũ và chu kỳ của `str2`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2
a1 = Solution().generateString("FF", "ab")
a2 = Solution().generateString("FFF", "aa")
assert a1 == generate_brute("FF", "ab") == "aaa"
assert a2 == generate_brute("FFF", "aa") == "abab"
print('Bài 1:', a1, '| Bài 2:', a2)

Bài 1: aaa | Bài 2: abab


In [22]:
# Bài 3, 4
def count_generated(s1, s2):
    n, m = len(s1), len(s2)
    L = n + m - 1
    w, _ = place_T(s1, s2)
    if w is None:
        return 0
    step = make_step(s2)
    forbid = [False] * L
    for i, c in enumerate(s1):
        if c == 'F':
            forbid[i + m - 1] = True
    cand = [(w[p],) if w[p] is not None else ('a', 'b') for p in range(L)]
    cur = {0: 1}
    for p in range(L):
        nxt = {}
        for st, ways in cur.items():
            for ch in cand[p]:
                ns = step(st, ch)
                if ns == m and forbid[p]:
                    continue
                nxt[ns] = nxt.get(ns, 0) + ways
        cur = nxt
    return sum(cur.values())

def t_compatible(s1, s2):
    m = len(s2)
    pi = prefix_function(s2)
    periods = set()
    b = pi[m - 1]
    while b:
        periods.add(m - b)
        b = pi[b - 1]
    Ts = [i for i, c in enumerate(s1) if c == 'T']
    return all(bb - a >= m or (bb - a) in periods for a, bb in zip(Ts, Ts[1:]))

rng = random.Random(19)
for _ in range(600):
    n = rng.randint(1, 5); m = rng.randint(1, 3)
    if n + m - 1 > 7:
        continue
    s1 = ''.join(rng.choice('TF') for _ in range(n)); s2 = ''.join(rng.choice('ab') for _ in range(m))
    brute_cnt = sum(1 for t in itertools.product('ab', repeat=n + m - 1) if is_generated(''.join(t), s1, s2))
    assert count_generated(s1, s2) == brute_cnt
    assert t_compatible(s1, s2) == (place_T(s1, s2)[0] is not None)
print('Bài 3, 4 khớp vét cạn. Ví dụ 1 có', count_generated("TFTF", "ab"), 'xâu trên {a, b}.')

Bài 3, 4 khớp vét cạn. Ví dụ 1 có 2 xâu trên {a, b}.


In [23]:
# Bài 5 — kiểm tra giả thuyết bằng máy
def check_hypothesis(trials, seed=5):
    rng = random.Random(seed)
    violations = 0
    for _ in range(trials):
        n = rng.randint(1, 30); m = rng.randint(1, 6)
        s1 = ''.join(rng.choice('TFF') for _ in range(n))
        s2 = ''.join(rng.choice('ab') for _ in range(m))
        w, _ = place_T(s1, s2)
        if w is None:
            continue
        free = [c is None for c in w]
        w = [c if c is not None else 'a' for c in w]
        for i, c in enumerate(s1):
            if c == 'F' and ''.join(w[i:i + m]) == s2:
                j = next((j for j in range(i + m - 1, i - 1, -1) if free[j]), None)
                if j is None:
                    break
                w[j] = 'b'; free[j] = False
                if any(s1[t] == 'F' and ''.join(w[t:t + m]) == s2 for t in range(i)):
                    violations += 1
    return violations

print('Số lần một cửa sổ F phía trước bị phá sau khi đổi:', check_hypothesis(20000))

Số lần một cửa sổ F phía trước bị phá sau khi đổi: 0


Kết quả `0` trên 20.000 bộ ngẫu nhiên ủng hộ giả thuyết. Hướng chứng minh (xem mục 10): bước 1 loại trường hợp cửa sổ cũ từng trùng; ở bước 2, đặt `d = i − i'`, hai cửa sổ cùng bằng `str2` cho `str2[t] = str2[t + d]` trên phần giao trừ đúng chỗ `t = j − i` (`'a'` so với `'b'`); các vị trí bên phải `j` đều do cửa sổ `T` bắt đầu sau `j` cố định, và chính các cửa sổ `T` đó buộc tính chu kỳ lan tới cả `t = j − i` — mâu thuẫn. Viết chi tiết lập luận này là phần "mini challenge".

## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
w = [''] * (n + m - 1)
for i với str1[i] == 'T': ghi str2 vào w[i..i+m-1], mâu thuẫn -> ''
free = [w[p] == '']; điền 'a' vào chỗ trống
for i với str1[i] == 'F':
    if w[i:i+m] == str2:
        j = vị trí free lớn nhất trong [i, i+m-1]; không có -> ''
        w[j] = 'b'; free[j] = False
return ''.join(w)
```

**Checklist:**

- [ ] Kiểm tra mâu thuẫn khi đặt `T`?
- [ ] Đổi vị trí tự do **phải nhất**, thành `'b'`?
- [ ] Chỉ đổi vị trí **tự do** (không đụng ký tự do `T` cố định), và đánh dấu đã đổi?
- [ ] Cửa sổ `F` không có vị trí tự do mà vẫn trùng ⇒ `""`?
- [ ] Độ dài kết quả `n + m − 1`?